# 01. Tool Server 实战

Tool 是 MCP 最核心的原语。一个 Tool Server 需要：

1. 声明工具列表（`tools/list`）：返回每个工具的 name、description、inputSchema
2. 响应工具调用（`tools/call`）：接收 name + arguments，返回 content[]
3. 错误处理：工具不存在、参数校验失败、执行超时

> **检查点**：能写出一个完整的 Tool 声明（JSON Schema inputSchema）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 定义工具注册与调用
from dataclasses import dataclass

@dataclass
class Tool:
    name: str; description: str; inputSchema: dict
    def call(self, arguments: dict) -> list[dict]: pass

class Calculator(Tool):
    def __init__(self):
        super().__init__("calculator", "Evaluate math",
            {"type":"object","properties":{"expr":{"type":"string"}},"required":["expr"]})
    def call(self, args):
        allowed = set("0123456789+-*/().% ")
        if not all(c in allowed for c in args["expr"]):
            return [{"type":"text","text":"Error: invalid chars"}]
        try:
            return [{"type":"text","text":str(eval(args["expr"],{"__builtins__":{}},{}))}]
        except Exception as e:
            return [{"type":"text","text":f"Error: {e}"}]

class Translator(Tool):
    def __init__(self):
        super().__init__("translator", "Translate text",
            {"type":"object","properties":{"text":{"type":"string"},"target":{"type":"string"}},"required":["text","target"]})
    def call(self, args):
        translations = {("hello","zh"):"你好",("hello","ja"):"こんにちは"}
        result = translations.get((args["text"].lower(),args["target"]), f"[{args['target']}] {args['text']}")
        return [{"type":"text","text":result}]

# 注册并测试
registry = {t.name: t for t in [Calculator(), Translator()]}
print("Tools:", list(registry.keys()))
print()
for name, test_args in [("calculator",{"expr":"(1+2)*3"}),("translator",{"text":"hello","target":"zh"})]:
    tool = registry[name]
    result = tool.call(test_args)
    print(f"{name}({test_args}) -> {result[0]['text']}")

Tools: ['calculator', 'translator']

calculator({'expr': '(1+2)*3'}) -> 9
translator({'text': 'hello', 'target': 'zh'}) -> 你好


## Tool Schema 设计原则

1. **description 要具体**：不是 `"Process data"`，而是 `"Calculate the average of a numeric array"`
2. **参数类型要严格**：用 `type` + `enum` + `minimum/maximum` 限制范围
3. **错误信息要可操作**：告诉 Agent 哪里错了、怎么修正，而非只返回 `"Error"`

> **检查点**：能为一个业务场景设计 3 个 Tool 的完整 inputSchema。

## 真实 API 实验

MCP 的 tools/call 与 OpenAI function calling 是同构的。下面用真实 function calling 模拟「MCP host」行为：schema 来自 MCP 风格定义。


In [3]:
# MCP 风格 schema → OpenAI tools 格式 → 真实调用
mcp_tool = {"name": "weather", "description": "查询城市天气", "inputSchema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}

# MCP host 的核心工作之一：协议转换
openai_tool = {"type": "function", "function": {"name": mcp_tool["name"], "description": mcp_tool["description"], "parameters": mcp_tool["inputSchema"]}}

def weather(args):
    return f"{args['city']}：晴，22°C（模拟 MCP server 返回）"

messages = [{"role": "user", "content": "上海今天天气怎么样？"}]
final, trace = chat_with_tools(client, messages, [openai_tool], {"weather": weather}, verbose=True)
show_trace(trace, "MCP 风格工具的真实调用")
print("\n回答:", final)


  🔧 weather({'city': '上海'}) → 上海：晴，22°C（模拟 MCP server 返回）


=== MCP 风格工具的真实调用 (1 calls) ===
[1] weather({'city': '上海'})
    → 上海：晴，22°C（模拟 MCP server 返回）

回答: 上海今天天气不错哦！具体情况如下：

🌤 **天气：** 晴
🌡 **温度：** 22°C

是个晴朗的好天气，温度适宜，非常适合出门活动！不过早晚可能会稍凉一些，建议带件薄外套。祝您愉快！😊


## 练习

1. 新增一个 `FileReader` 工具：接受 `path` 参数，返回文件内容。增加安全检查（路径穿越防护）。
2. 实现 `tools/call` 的超时机制：如果工具执行超过 5 秒，自动返回 timeout error。
3. 设计一个 Tool 的版本管理策略：当 inputSchema 变化时，如何保持向后兼容？

> **检查点**：能为你的 MCP server 实现 tools/list 和 tools/call 两个端点，并正确处理错误。